<a href="https://colab.research.google.com/github/yuhui-0611/Unstructured-Data-Analysis/blob/main/UDA_project_Data_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# API 전

In [52]:
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

base = Path('/content/drive/MyDrive/UDA-Project')
data_path = base / 'data'
save_path = base / 'processed'
save_path.mkdir(exist_ok=True)

names = ['형신', '윤주', '유희']
dfs = []

for name in names:
    file = data_path / f'shorts_metadata_{name}.csv'

    # 인코딩 자동 확인
    try:
        df = pd.read_csv(file, dtype=str, encoding='utf-8-sig')
        print(f'{name}: UTF-8로 읽기 성공')
    except UnicodeDecodeError:
        df = pd.read_csv(file, dtype=str, encoding='cp949')
        print(f'{name}: CP949로 읽기 성공')

    df['source'] = name
    dfs.append(df)

# 세 파일 병합
merged = pd.concat(dfs, ignore_index=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
형신: UTF-8로 읽기 성공
윤주: UTF-8로 읽기 성공
유희: UTF-8로 읽기 성공


In [53]:
# 1. 데이터 크기 및 컬럼명
print('데이터 크기:', merged.shape)
print('\n컬럼명:')
print(merged.columns.tolist())

# 2. 컬럼별 결측치
print('\n[결측치 현황]')
display(merged.isna().sum().to_frame('결측치 수'))

# 3. 완전히 동일한 행
print('\n[중복 행]')
print(merged.duplicated().sum())

# 4. 팀원별 데이터 수
print('\n[팀원별 데이터 수]')
display(merged['source'].value_counts())

데이터 크기: (1491, 26)

컬럼명:
['video_id', 'url', 'channel', 'channel_id', 'channel_url', 'channel_follower_count', 'title', 'description', 'tags', 'title_hashtags', 'desc_hashtags', 'drama_guess', 'drama_name', 'uploaded_at', 'collected_at', 'days_since_upload', 'duration_sec', 'view_count', 'like_count', 'comment_count', 'width', 'height', 'vcodec', 'video_file', 'source', 'Unnamed: 23']

[결측치 현황]


,결측치 수
video_id,0
url,0
channel,0
channel_id,0
channel_url,0
channel_follower_count,1042
title,0
description,113
tags,1372
title_hashtags,206



[중복 행]
0

[팀원별 데이터 수]


,count
source,
유희,635
윤주,449
형신,407


In [54]:
# 1. 드라마명 종류 및 개수
print('===== 드라마명 목록 =====')
display(
    merged['drama_name']
    .value_counts(dropna=False)
    .to_frame('영상 수')
)

# 2. 드라마명이 비어 있는 영상
print('\n===== 드라마명 결측 영상 =====')
display(
    merged.loc[
        merged['drama_name'].isna(),
        ['video_id', 'title', 'drama_guess', 'source']
    ]
)

# 3. 영상 ID 중복 확인
print('\n===== 영상 ID 중복 =====')
print('중복 ID 수:', merged['video_id'].duplicated().sum())

display(
    merged.loc[
        merged['video_id'].duplicated(keep=False),
        ['video_id', 'title', 'drama_name', 'source']
    ].sort_values('video_id')
)

# 4. 영상 ID 형식 확인
print('\n===== 영상 ID 형식 이상 =====')

invalid_id = ~merged['video_id'].str.fullmatch(
    r'[A-Za-z0-9_-]{11}', na=False
)

display(
    merged.loc[
        invalid_id,
        ['video_id', 'url', 'title', 'source']
    ]
)

# 5. 주요 메타데이터 결측 영상
print('\n===== 주요 메타데이터 결측 =====')

important_cols = [
    'uploaded_at', 'collected_at',
    'duration_sec', 'view_count',
    'width', 'height', 'video_file'
]

display(
    merged.loc[
        merged[important_cols].isna().any(axis=1),
        ['video_id', 'title', 'source'] + important_cols
    ]
)

===== 드라마명 목록 =====


,영상 수
drama_name,
미스터션샤인,149
포핸즈,87
날아올라라나비,84
부부의세계,80
모두가 자신의 무가치함과 싸우고 있다,64
...,...
유부녀 킬러,1
뉴노멀,1
슬픈 열대,1



===== 드라마명 결측 영상 =====


,video_id,title,drama_guess,source
0,XO4W2djld2g,"영어 시간에 국어 질문하는 학생, 결국… | #shorts #쇼츠 #드라마 #영화",영화,형신
39,Kc2awsgivdQ,의외로 세상에는 별게 다 있음 | #shorts #쇼츠 #정보 #꿀팁 #Ai #광고,정보,형신
40,sEvRDQPmXk0,드디어 올 것이 왔다… | #shorts #쇼츠 #꿀팁 #정보 #다이소 #신상,꿀팁,형신
48,qNmnmdHQofc,써보면 영원히 쓰는 것들 | #shorts #쇼츠 #꿀팁 #광고,꿀팁,형신
87,DVK0U2E7HKo,"여름에 바다,수영장 안 갈거야? | #몸무게계산 #비피더스다이어트유산균 #이장우다이...",몸무게계산,형신
92,0WQZGhkUT7Q,고양이가 댄스 챌린지 하는 법… | #shorts #고양이 #동물 #챌린지,고양이,형신
96,NCQH5hVAG4w,‘매끈매끈 푱푱한‘ 카니의 찐텐 풀 장착 모먼트 | #shorts #오늘의집 #카니,오늘의집,형신
97,wz_gMPVgJRQ,‘비X고’ 의 창시자도 참을 수 없는 ‘이 것‘ / #shorts #쇼츠 #오늘의집...,오늘의집,형신



===== 영상 ID 중복 =====
중복 ID 수: 15


,video_id,title,drama_name,source
31,#NAME?,하다하다 딸 전세금까지 가져가는 엄마 | #shorts #쇼츠 #드라마 #영화 #정...,정신병동에도 아침이 와요,형신
45,#NAME?,상남자 최영도의 진상 처리 법… | #shorts #쇼츠 #드라마 #영화 #상속자들,상속자들,형신
95,#NAME?,#디즈니플러스 #디플 #ott추천 #디즈니플러스추천#북극성 #전지현 #강동원,북극성,형신
209,#NAME?,병원에서 만난 남미새 집으로 부른 여간호사 뜨거운 밤은? #넷플릭스 #새콤달콤 #채...,새콤달콤,형신
253,#NAME?,애마부인 여주인공 노출 캐스팅의 숨은 진실 #넷플릭스 #애마 #이하늬 #방효린 #드...,애마,형신
314,#NAME?,이유미 심쿵하게 만든 우도환의 돌직구 고백 #넷플릭스 #우도환 #이유미 #오정세 #...,Mr.플랑크톤,형신
448,#NAME?,단발 해달라던 여자 손님의 역대급 뻔뻔한 행동 #shorts #날아올라라나비,날아올라라나비,윤주
457,#NAME?,염색 다시 해달라고 찾아온 진상인 줄 알았던 손님의 반전 #shorts #날아올라라나비,날아올라라나비,윤주
467,#NAME?,동생 짜장면 한 젓가락 뺏어 먹으려던 여자의 최후 #shorts #날아올라라나비,날아올라라나비,윤주
479,#NAME?,이런 식이면 다른 미용실처럼 복장 체크를 할 수 밖에 없어ㅋ #shorts #날아올...,날아올라라나비,윤주



===== 영상 ID 형식 이상 =====


,video_id,url,title,source
31,#NAME?,https://www.youtube.com/shorts/-XY7mqap820,하다하다 딸 전세금까지 가져가는 엄마 | #shorts #쇼츠 #드라마 #영화 #정...,형신
45,#NAME?,https://www.youtube.com/shorts/-grJHiRecRM,상남자 최영도의 진상 처리 법… | #shorts #쇼츠 #드라마 #영화 #상속자들,형신
95,#NAME?,https://www.youtube.com/shorts/-uefvTFvbcU,#디즈니플러스 #디플 #ott추천 #디즈니플러스추천#북극성 #전지현 #강동원,형신
209,#NAME?,https://www.youtube.com/shorts/-HrxGAbCl5M,병원에서 만난 남미새 집으로 부른 여간호사 뜨거운 밤은? #넷플릭스 #새콤달콤 #채...,형신
253,#NAME?,https://www.youtube.com/shorts/-rbjhouLhJI,애마부인 여주인공 노출 캐스팅의 숨은 진실 #넷플릭스 #애마 #이하늬 #방효린 #드...,형신
314,#NAME?,https://www.youtube.com/shorts/-v094SA3NL8,이유미 심쿵하게 만든 우도환의 돌직구 고백 #넷플릭스 #우도환 #이유미 #오정세 #...,형신
448,#NAME?,https://www.youtube.com/shorts/-WglXBJjohs,단발 해달라던 여자 손님의 역대급 뻔뻔한 행동 #shorts #날아올라라나비,윤주
457,#NAME?,https://www.youtube.com/shorts/-ASrjtlQFzs,염색 다시 해달라고 찾아온 진상인 줄 알았던 손님의 반전 #shorts #날아올라라나비,윤주
467,#NAME?,https://www.youtube.com/shorts/-FHtIw7nlHQ,동생 짜장면 한 젓가락 뺏어 먹으려던 여자의 최후 #shorts #날아올라라나비,윤주
479,#NAME?,https://www.youtube.com/shorts/-Bndh1nrbX4,이런 식이면 다른 미용실처럼 복장 체크를 할 수 밖에 없어ㅋ #shorts #날아올...,윤주



===== 주요 메타데이터 결측 =====


,video_id,title,source,uploaded_at,collected_at,duration_sec,view_count,width,height,video_file
89,0tkNsqYf0cY,이른바 군대병ㅋㅋ,형신,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 영상 ID 복구 및 불필요한 컬럼 제거

In [55]:
# 1. URL에서 영상 ID 추출
url_id = merged['url'].str.extract(
    r'(?:shorts/|watch\?v=|youtu\.be/)([A-Za-z0-9_-]{11})'
)[0]

# 2. 손상된 영상 ID만 복구
invalid = ~merged['video_id'].str.fullmatch(
    r'[A-Za-z0-9_-]{11}', na=False
)

merged.loc[invalid, 'video_id'] = url_id[invalid]

print('복구 대상:', invalid.sum())
print('복구 성공:', merged.loc[invalid, 'video_id'].notna().sum())

# 3. 불필요한 컬럼 제거
merged = merged.loc[
    :, ~merged.columns.str.startswith('Unnamed:')
]

# 4. 영상 ID 재검사
invalid_after = ~merged['video_id'].str.fullmatch(
    r'[A-Za-z0-9_-]{11}', na=False
)

print('\n남은 비정상 ID:', invalid_after.sum())
print('중복 영상 ID:', merged['video_id'].duplicated().sum())

print('데이터 크기:', merged.shape)

복구 대상: 16
복구 성공: 16

남은 비정상 ID: 0
중복 영상 ID: 0
데이터 크기: (1491, 25)


### 드라마명 중복 확인

In [56]:
# 드라마명 목록과 영상 수 확인
drama_counts = (
    merged['drama_name']
    .value_counts(dropna=False)
    .reset_index()
)

drama_counts.columns = ['drama_name', 'count']

print('드라마명 종류:', merged['drama_name'].nunique())
display(drama_counts)

드라마명 종류: 121


,drama_name,count
0,미스터션샤인,149
1,포핸즈,87
2,날아올라라나비,84
3,부부의세계,80
4,모두가 자신의 무가치함과 싸우고 있다,64
...,...,...
117,유부녀 킬러,1
118,뉴노멀,1
119,슬픈 열대,1
120,하트 호브 비스트,1


- 100일의 거짓말
- D.P.
- 살인자o난감
- 이 사랑 통역 되나요?
- 파이널 테이블

> 이걸로 통일

In [57]:
# 드라마명 표기 통일
name_map = {
    '100일의거짓말': '100일의 거짓말',
    'DP': 'D.P.',
    '살인자ㅇ난감': '살인자o난감',  # 영문 소문자 o
    '살인자O난감': '살인자o난감',
    '이사랑 통역되나요?': '이 사랑 통역 되나요?',
    '이 사랑 통역되나요?': '이 사랑 통역 되나요?',
    '파이널테이블': '파이널 테이블'
}

# 기존 값의 앞뒤 공백 제거 후 통일
merged['drama_name'] = (
    merged['drama_name']
    .str.strip()
    .replace(name_map)
)

# 결과 확인
print('드라마명 종류:', merged['drama_name'].nunique())
display(merged['drama_name'].value_counts())

드라마명 종류: 117


,count
drama_name,
미스터션샤인,149
포핸즈,87
날아올라라나비,84
부부의세계,80
100일의 거짓말,65
...,...
유부녀 킬러,1
뉴노멀,1
슬픈 열대,1


In [58]:
# 드라마명이 없는 영상 확인 - 8개
missing_drama = merged[
    merged['drama_name'].isna()
]

display(
    missing_drama[
        ['video_id', 'title', 'drama_guess', 'url', 'source']
    ]
)

,video_id,title,drama_guess,url,source
0,XO4W2djld2g,"영어 시간에 국어 질문하는 학생, 결국… | #shorts #쇼츠 #드라마 #영화",영화,https://www.youtube.com/shorts/XO4W2djld2g,형신
39,Kc2awsgivdQ,의외로 세상에는 별게 다 있음 | #shorts #쇼츠 #정보 #꿀팁 #Ai #광고,정보,https://www.youtube.com/shorts/Kc2awsgivdQ,형신
40,sEvRDQPmXk0,드디어 올 것이 왔다… | #shorts #쇼츠 #꿀팁 #정보 #다이소 #신상,꿀팁,https://www.youtube.com/shorts/sEvRDQPmXk0,형신
48,qNmnmdHQofc,써보면 영원히 쓰는 것들 | #shorts #쇼츠 #꿀팁 #광고,꿀팁,https://www.youtube.com/shorts/qNmnmdHQofc,형신
87,DVK0U2E7HKo,"여름에 바다,수영장 안 갈거야? | #몸무게계산 #비피더스다이어트유산균 #이장우다이...",몸무게계산,https://www.youtube.com/shorts/DVK0U2E7HKo,형신
92,0WQZGhkUT7Q,고양이가 댄스 챌린지 하는 법… | #shorts #고양이 #동물 #챌린지,고양이,https://www.youtube.com/shorts/0WQZGhkUT7Q,형신
96,NCQH5hVAG4w,‘매끈매끈 푱푱한‘ 카니의 찐텐 풀 장착 모먼트 | #shorts #오늘의집 #카니,오늘의집,https://www.youtube.com/shorts/NCQH5hVAG4w,형신
97,wz_gMPVgJRQ,‘비X고’ 의 창시자도 참을 수 없는 ‘이 것‘ / #shorts #쇼츠 #오늘의집...,오늘의집,https://www.youtube.com/shorts/wz_gMPVgJRQ,형신


### 드라마명 결측 제거

In [59]:
# 1. 삭제 전 데이터 수
print('삭제 전:', len(merged))

# 2. drama_name이 NaN인 행 삭제
merged = merged.dropna(subset=['drama_name']).reset_index(drop=True)

# 3. 삭제 후 데이터 수
print('삭제 후:', len(merged))
print('남은 결측치:', merged['drama_name'].isna().sum())

# 4. 전처리 결과 저장
# merged.to_csv(
#     save_path / 'shorts_preprocessed.csv',
#     index=False,
#     encoding='utf-8-sig'
# )

# print('저장 완료!')

삭제 전: 1491
삭제 후: 1483
남은 결측치: 0


## 최종 결측 점검

In [60]:
# 1. 전체 데이터 확인
print('전체 영상 수:', len(merged))

# 2. 영상 ID 형식 검사
invalid_id = ~merged['video_id'].str.fullmatch(
    r'[A-Za-z0-9_-]{11}', na=False
)

print('\n[영상 ID 형식 이상]')
print('이상 ID 수:', invalid_id.sum())
display(merged.loc[invalid_id, ['video_id', 'url', 'title']])

# 3. 영상 ID 중복 검사
duplicate_id = merged['video_id'].duplicated(keep=False)

print('\n[영상 ID 중복]')
print('중복 영상 행 수:', duplicate_id.sum())
display(
    merged.loc[
        duplicate_id,
        ['video_id', 'title', 'source']
    ].sort_values('video_id')
)

# 4. 전체 결측치 확인
print('\n[컬럼별 결측치]')
display(
    merged.isna().sum().to_frame('결측치 수')
)

# 5. 주요 메타데이터 결측 영상 확인
important_cols = [
    'uploaded_at', 'duration_sec',
    'view_count', 'width', 'height', 'video_file'
]

print('\n[주요 메타데이터 결측 영상]')
display(
    merged.loc[
        merged[important_cols].isna().any(axis=1),
        ['video_id', 'title', 'source'] + important_cols
    ]
)

전체 영상 수: 1483

[영상 ID 형식 이상]
이상 ID 수: 0


,video_id,url,title



[영상 ID 중복]
중복 영상 행 수: 0


,video_id,title,source



[컬럼별 결측치]


,결측치 수
video_id,0
url,0
channel,0
channel_id,0
channel_url,0
channel_follower_count,1034
title,0
description,110
tags,1370
title_hashtags,206



[주요 메타데이터 결측 영상]


,video_id,title,source,uploaded_at,duration_sec,view_count,width,height,video_file
84,0tkNsqYf0cY,이른바 군대병ㅋㅋ,형신,NaN,NaN,NaN,NaN,NaN,NaN


# API 수집

In [61]:
import requests
from google.colab import userdata

# 1. Colab에 저장한 API 키 불러오기
API_KEY = userdata.get('YOUTUBE_API_KEY')

# 2. 영상 하나 선택
video_id = merged['video_id'].iloc[0]

# 3. YouTube API에 요청
response = requests.get(
    'https://www.googleapis.com/youtube/v3/videos',
    params={
        'part': 'snippet,statistics',
        'id': video_id,
        'key': API_KEY
    },
    timeout=30
)

# 4. 오류 확인
response.raise_for_status()

# 5. 결과 확인
data = response.json()

if data.get('items'):
    video = data['items'][0]
    print('영상 제목:', video['snippet']['title'])
    print('조회수:', video['statistics'].get('viewCount'))
    print('좋아요:', video['statistics'].get('likeCount'))
    print('댓글 수:', video['statistics'].get('commentCount'))
else:
    print('API에서 조회되지 않는 영상입니다.')

영상 제목: 밥솥 뚜껑이 고정 안되어있으면 이렇게 됨… | #shorts #쇼츠 #드라마 #영화
조회수: 19445
좋아요: 134
댓글 수: 3


- 위는 예시로 아무 영상 돌려본거
- 코랩에 미리 API 키 저장해뒀기에 다음에 또 키 불러올 필요 X

< 코드 설명 >
- API_KEY: API 사용 권한을 확인하는 키
- video_id: 조회할 유튜브 영상 ID
- requests.get(): 유튜브 서버에 정보 요청
- statistics: 조회수·좋아요·댓글 수 정보
- response.json(): 서버의 응답을 파이썬에서 사용할 수 있는 형태로 변환

In [62]:
import pandas as pd
import requests
from datetime import datetime, timezone
from google.colab import userdata

# 1. API 키 불러오기
API_KEY = userdata.get('YOUTUBE_API_KEY')

# 2. 영상 ID 목록
video_ids = merged['video_id'].tolist()
results = []

# 3. 50개씩 나눠서 API 호출
for i in range(0, len(video_ids), 50):
    batch = video_ids[i:i+50]

    response = requests.get(
        'https://www.googleapis.com/youtube/v3/videos',
        params={
            'part': 'statistics,snippet,contentDetails',
            'id': ','.join(batch),
            'key': API_KEY
        },
        timeout=30
    )
    response.raise_for_status()

    collected_time = datetime.now(timezone.utc).isoformat()

    for item in response.json().get('items', []):
        stats = item.get('statistics', {})
        snippet = item.get('snippet', {})
        details = item.get('contentDetails', {})

        results.append({
            'video_id': item['id'],
            'view_count_new': stats.get('viewCount'),
            'like_count_new': stats.get('likeCount'),
            'comment_count_new': stats.get('commentCount'),
            'uploaded_at_new': snippet.get('publishedAt'),
            'duration_api': details.get('duration'),
            'collected_at_new': collected_time
        })

    print(f'{min(i+50, len(video_ids))} / {len(video_ids)} 완료')

# 4. API 결과를 데이터프레임으로 변환
api_df = pd.DataFrame(results)

# 5. 숫자형으로 변환
for col in ['view_count_new', 'like_count_new', 'comment_count_new']:
    api_df[col] = pd.to_numeric(api_df[col], errors='coerce').astype('Int64')

print('\nAPI 재수집 완료!')
print('전체 영상:', len(video_ids))
print('API 조회 성공:', len(api_df))
print('API 조회 실패:', len(video_ids) - len(api_df))

display(api_df.head())

50 / 1483 완료
100 / 1483 완료
150 / 1483 완료
200 / 1483 완료
250 / 1483 완료
300 / 1483 완료
350 / 1483 완료
400 / 1483 완료
450 / 1483 완료
500 / 1483 완료
550 / 1483 완료
600 / 1483 완료
650 / 1483 완료
700 / 1483 완료
750 / 1483 완료
800 / 1483 완료
850 / 1483 완료
900 / 1483 완료
950 / 1483 완료
1000 / 1483 완료
1050 / 1483 완료
1100 / 1483 완료
1150 / 1483 완료
1200 / 1483 완료
1250 / 1483 완료
1300 / 1483 완료
1350 / 1483 완료
1400 / 1483 완료
1450 / 1483 완료
1483 / 1483 완료

API 재수집 완료!
전체 영상: 1483
API 조회 성공: 1468
API 조회 실패: 15


,video_id,view_count_new,like_count_new,comment_count_new,uploaded_at_new,duration_api,collected_at_new
0,_PL_l6Nk83w,19445,134,3,2023-10-06T11:01:51Z,PT25S,2026-11-09T06:46:13.228438+00:00
1,gW9hbpBaBBY,11617,146,0,2023-10-07T11:02:16Z,PT28S,2026-11-09T06:46:13.228438+00:00
2,xka45k9WDc4,10561,129,2,2023-10-10T11:00:48Z,PT59S,2026-11-09T06:46:13.228438+00:00
3,NQt0VkR9q_0,134933,2631,21,2023-10-11T11:04:56Z,PT46S,2026-11-09T06:46:13.228438+00:00
4,ExRxa0wD3j4,16960,252,3,2023-10-12T11:06:04Z,PT44S,2026-11-09T06:46:13.228438+00:00


- YouTube Data API는 한 번에 영상 ID를 최대 50개씩 요청 가능
- 따라서 1,483개 영상은 총 30번 호출

In [63]:
# 1. 조회되지 않은 영상 확인
missing_ids = merged[
    ~merged['video_id'].isin(api_df['video_id'])
]

print('===== API 조회 실패 영상 =====')
print('조회 실패:', len(missing_ids))
display(missing_ids[['video_id', 'title', 'url', 'source']])

# 2. 재수집한 지표의 결측치 확인
print('\n===== API 지표 결측치 =====')
display(
    api_df[
        ['view_count_new', 'like_count_new', 'comment_count_new']
    ].isna().sum().to_frame('결측치 수')
)

# 3. 수집 시간 확인
print('\n===== 재수집 시간 (UTC) =====')
print('시작:', api_df['collected_at_new'].min() if len(api_df) else '없음')
print('종료:', api_df['collected_at_new'].max() if len(api_df) else '없음')

===== API 조회 실패 영상 =====
조회 실패: 15


,video_id,title,url,source
124,cXkSG31yYnk,신병2 특급전사를 구별하고 개빡친 중대장#신병#신병2#중대장#특급전사#드라마#드라마쇼츠,https://www.youtube.com/shorts/cXkSG31yYnk,형신
125,19r-LEE74rY,신병2 중대장부조리 마음의편지를 본 중대장 법인에정체는?1편#신병#신병2#드라마#범...,https://www.youtube.com/shorts/19r-LEE74rY,형신
127,idF1fIBhV3g,신병2 중대장 갑질에 행보관도 분노해서 과거군대 이야기 까지..마음에편지의범인은? ...,https://www.youtube.com/shorts/idF1fIBhV3g,형신
143,zbB3_fAr5bM,사냥개들2 빌런 백정 WBC챔피언도 잔인하게 죽음까지 몰아넣고..#사냥개들2#정지훈...,https://www.youtube.com/shorts/zbB3_fAr5bM,형신
144,JkfSaGVwpIE,사냥개들2 감옥에서 탈출한 인범은 도망갈려고 하지만 백정한테 결국...#사냥개들2#...,https://www.youtube.com/shorts/JkfSaGVwpIE,형신
151,vFIpKbxXmPA,일진과 바람핀 유부남 교사 왕따가 복수하는방법은?#이은샘#아린#S라인,https://www.youtube.com/shorts/vFIpKbxXmPA,형신
179,PpTy-dnArw4,원더풀스 연쇄살인마 박은빈과 초능력자 차은우와 만남은? #원더풀스 #박은빈 #차은우...,https://www.youtube.com/shorts/PpTy-dnArw4,형신
194,cnXiSyEu-5g,신병3 말년 병장 간식 트윅스를 조작을 하는 분대원 들은? 2편 #신병 #신병3 ...,https://www.youtube.com/shorts/cnXiSyEu-5g,형신
195,sgx3m6pFG08,신병3 최일구 간식 사고 사건 맞선임 배신으로 종결 #신병 #신병3 #드라마 #숏츠...,https://www.youtube.com/shorts/sgx3m6pFG08,형신
201,-HrxGAbCl5M,병원에서 만난 남미새 집으로 부른 여간호사 뜨거운 밤은? #넷플릭스 #새콤달콤 #채...,https://www.youtube.com/shorts/-HrxGAbCl5M,형신



===== API 지표 결측치 =====


,결측치 수
view_count_new,0
like_count_new,0
comment_count_new,1



===== 재수집 시간 (UTC) =====
시작: 2026-11-09T06:46:13.228438+00:00
종료: 2026-11-09T06:46:17.419425+00:00


## merge

In [64]:
# 1. 기존 데이터와 API 결과 합치기
new_cols = [col for col in api_df.columns if col != 'video_id']

# 재실행 시 중복 컬럼 방지
merged = merged.drop(columns=new_cols, errors='ignore')

merged = merged.merge(
    api_df,
    on='video_id',
    how='left',
    validate='one_to_one'
)

# 2. API 조회 성공 여부 표시
merged['api_found'] = merged['video_id'].isin(api_df['video_id'])

# 3. 결합 결과 확인
print('전체 영상:', len(merged))
print('API 조회 성공:', merged['api_found'].sum())
print('API 조회 실패:', (~merged['api_found']).sum())

print('\n재수집 지표 결측치:')
display(
    merged[
        ['view_count_new', 'like_count_new', 'comment_count_new']
    ].isna().sum()
)

# 4. 댓글 수가 없는 영상 확인
print('\n댓글 수 결측 영상:')
display(
    merged.loc[
        merged['api_found'] & merged['comment_count_new'].isna(),
        ['video_id', 'title', 'url', 'source']
    ]
)

전체 영상: 1483
API 조회 성공: 1468
API 조회 실패: 15

재수집 지표 결측치:


,0
view_count_new,15
like_count_new,15
comment_count_new,16



댓글 수 결측 영상:


,video_id,title,url,source
17,l4weA5YIG4E,정신병동에 입원한 환자가 간호사 한 명에게 꽂혔을 때 | #shorts #쇼츠 #드...,https://www.youtube.com/shorts/l4weA5YIG4E,형신


- API 조회 성공 1,468개: 재수집 지표 사용
- API 조회 실패 15개: 기존 데이터는 유지하되 재수집 지표는 NaN
- 댓글 수 결측 영상 1개: NaN 유지
- 기존 지표: 비교 및 확인용으로 보존

In [65]:
# 1. 기존 지표 이름 변경
rename_old = {
    'view_count': 'view_count_original',
    'like_count': 'like_count_original',
    'comment_count': 'comment_count_original',
    'collected_at': 'collected_at_original'
}

merged = merged.rename(columns=rename_old)

# 2. 재수집 지표 이름 변경
rename_new = {
    'view_count_new': 'view_count',
    'like_count_new': 'like_count',
    'comment_count_new': 'comment_count',
    'collected_at_new': 'collected_at'
}

merged = merged.rename(columns=rename_new)

# 3. 결과 확인
display(
    merged[
        ['video_id', 'view_count', 'like_count',
         'comment_count', 'collected_at', 'api_found']
    ].head()
)

print('\n재수집 지표 결측치:')
display(
    merged[
        ['view_count', 'like_count', 'comment_count']
    ].isna().sum()
)

,video_id,view_count,like_count,comment_count,collected_at,api_found
0,_PL_l6Nk83w,19445,134,3,2026-11-09T06:46:13.228438+00:00,True
1,gW9hbpBaBBY,11617,146,0,2026-11-09T06:46:13.228438+00:00,True
2,xka45k9WDc4,10561,129,2,2026-11-09T06:46:13.228438+00:00,True
3,NQt0VkR9q_0,134933,2631,21,2026-11-09T06:46:13.228438+00:00,True
4,ExRxa0wD3j4,16960,252,3,2026-11-09T06:46:13.228438+00:00,True



재수집 지표 결측치:


,0
view_count,15
like_count,15
comment_count,16


- 기존 값에는 _original을 붙이고, API로 수집한 값을 기본 지표 컬럼으로 변경

## 업로드 날짜와 경과일 정리

- 기존 uploaded_at이 비어 있는 경우에는 API에서 가져온 uploaded_at_new로 채움
- days_since_upload는 새로운 API 수집 시각을 기준으로 다시 계산

In [66]:
# 1. 날짜 형식으로 변환
merged['uploaded_at'] = pd.to_datetime(
    merged['uploaded_at'], errors='coerce', utc=True
)

merged['uploaded_at_new'] = pd.to_datetime(
    merged['uploaded_at_new'], errors='coerce', utc=True
)

merged['collected_at'] = pd.to_datetime(
    merged['collected_at'], errors='coerce', utc=True
)

# 2. 업로드 날짜 결측치 보완
merged['uploaded_at'] = merged['uploaded_at'].fillna(
    merged['uploaded_at_new']
)

# 3. 업로드 경과일 재계산
merged['days_since_upload'] = (
    (merged['collected_at'] - merged['uploaded_at'])
    .dt.total_seconds() / 86400
)

# 4. 결과 확인
print('업로드 날짜 결측:', merged['uploaded_at'].isna().sum())
print('업로드 경과일 결측:', merged['days_since_upload'].isna().sum())

display(
    merged[
        ['video_id', 'uploaded_at',
         'collected_at', 'days_since_upload']
    ].head()
)

업로드 날짜 결측: 0
업로드 경과일 결측: 15


,video_id,uploaded_at,collected_at,days_since_upload
0,_PL_l6Nk83w,2023-10-06 11:01:51+00:00,2026-11-09 06:46:13.228438+00:00,1129.822479
1,gW9hbpBaBBY,2023-10-07 11:02:16+00:00,2026-11-09 06:46:13.228438+00:00,1128.822190
2,xka45k9WDc4,2023-10-10 11:00:48+00:00,2026-11-09 06:46:13.228438+00:00,1125.823209
3,NQt0VkR9q_0,2023-10-11 11:04:56+00:00,2026-11-09 06:46:13.228438+00:00,1124.820338
4,ExRxa0wD3j4,2023-10-12 11:06:04+00:00,2026-11-09 06:46:13.228438+00:00,1123.819551


## 영상 길이 결측치 보완

In [67]:
# 1. API 영상 길이를 초 단위로 변환
duration = pd.to_timedelta(
    merged['duration_api'], errors='coerce'
)

merged['duration_sec'] = pd.to_numeric(
    merged['duration_sec'], errors='coerce'
).fillna(duration.dt.total_seconds())

# 2. 결과 확인
print('영상 길이 결측:', merged['duration_sec'].isna().sum())

display(
    merged.loc[
        merged['duration_sec'].isna(),
        ['video_id', 'title', 'url', 'api_found']
    ]
)

영상 길이 결측: 0


,video_id,title,url,api_found


## 나머지 결측치 점검

In [68]:
# 남은 결측치 확인
print('===== 전체 결측치 =====')
display(
    merged.isna().sum().to_frame('결측치 수')
)

# 영상 파일 관련 결측치 확인
print('\n===== 영상 파일 관련 결측 =====')
cols = ['width', 'height', 'vcodec', 'video_file']

display(
    merged.loc[
        merged[cols].isna().any(axis=1),
        ['video_id', 'title', 'source', 'api_found'] + cols
    ]
)

# 채널 구독자 수 수집 현황
print('\n===== 팀원별 구독자 수 결측 =====')
display(
    merged.groupby('source')['channel_follower_count']
    .apply(lambda x: x.isna().sum())
)

===== 전체 결측치 =====


,결측치 수
video_id,0
url,0
channel,0
channel_id,0
channel_url,0
channel_follower_count,1034
title,0
description,110
tags,1370
title_hashtags,206



===== 영상 파일 관련 결측 =====


,video_id,title,source,api_found,width,height,vcodec,video_file
84,0tkNsqYf0cY,이른바 군대병ㅋㅋ,형신,True,NaN,NaN,NaN,NaN



===== 팀원별 구독자 수 결측 =====


,channel_follower_count
source,
유희,635
윤주,0
형신,399


- 구독자수 API 재수집

In [69]:
import pandas as pd
import requests
from datetime import datetime, timezone
from google.colab import userdata

# 1. API 키 불러오기
API_KEY = userdata.get('YOUTUBE_API_KEY')

# 2. 중복 없는 채널 ID 추출
channel_ids = merged['channel_id'].dropna().unique().tolist()
results = []

print('전체 영상:', len(merged))
print('고유 채널 수:', len(channel_ids))

# 3. 채널 50개씩 API 조회
for i in range(0, len(channel_ids), 50):
    batch = channel_ids[i:i+50]

    response = requests.get(
        'https://www.googleapis.com/youtube/v3/channels',
        params={
            'part': 'statistics',
            'id': ','.join(batch),
            'key': API_KEY
        },
        timeout=30
    )
    response.raise_for_status()

    collected_time = datetime.now(timezone.utc).isoformat()

    for item in response.json().get('items', []):
        stats = item.get('statistics', {})

        results.append({
            'channel_id': item['id'],
            'follower_count_new': (
                None if stats.get('hiddenSubscriberCount')
                else stats.get('subscriberCount')
            ),
            'follower_collected_at': collected_time
        })

    print(f'{min(i+50, len(channel_ids))}/{len(channel_ids)} 완료')

# 4. 결과 데이터프레임 생성
channel_df = pd.DataFrame(
    results,
    columns=['channel_id', 'follower_count_new',
             'follower_collected_at']
)

channel_df['follower_count_new'] = pd.to_numeric(
    channel_df['follower_count_new'], errors='coerce'
).astype('Int64')

# 5. 기존 데이터에 연결
merged = merged.drop(
    columns=['follower_count_new', 'follower_collected_at'],
    errors='ignore'
)

merged = merged.merge(
    channel_df,
    on='channel_id',
    how='left',
    validate='many_to_one'
)

print('\n재수집 완료!')
print('API 조회 성공 채널:', len(channel_df))
print('구독자 수 결측 영상:', merged['follower_count_new'].isna().sum())

display(
    merged[['channel', 'channel_id',
            'follower_count_new']].head()
)

전체 영상: 1483
고유 채널 수: 8
8/8 완료

재수집 완료!
API 조회 성공 채널: 8
구독자 수 결측 영상: 0


,channel,channel_id,follower_count_new
0,1분 샤샤샥,UCn8w4bbuPv4WUN4ZK3_JcpQ,20600
1,1분 샤샤샥,UCn8w4bbuPv4WUN4ZK3_JcpQ,20600
2,1분 샤샤샥,UCn8w4bbuPv4WUN4ZK3_JcpQ,20600
3,1분 샤샤샥,UCn8w4bbuPv4WUN4ZK3_JcpQ,20600
4,1분 샤샤샥,UCn8w4bbuPv4WUN4ZK3_JcpQ,20600


In [70]:
# 채널 조회에 실패한 경우 확인
missing_channels = [
    cid for cid in channel_ids
    if cid not in channel_df['channel_id'].tolist()
]

print('전체 고유 채널:', len(channel_ids))
print('API 조회 실패 채널:', len(missing_channels))

# 팀원별 구독자 수 결측
print('\n[팀원별 구독자 수 결측]')
display(
    merged.groupby('source')['follower_count_new']
    .apply(lambda x: x.isna().sum())
)

# 구독자 수가 없는 채널 확인
print('\n[구독자 수 결측 채널]')
display(
    merged.loc[
        merged['follower_count_new'].isna(),
        ['channel', 'channel_id']
    ].drop_duplicates()
)

전체 고유 채널: 8
API 조회 실패 채널: 0

[팀원별 구독자 수 결측]


,follower_count_new
source,
유희,0
윤주,0
형신,0



[구독자 수 결측 채널]


,channel,channel_id


In [71]:
# 기존 구독자 수 보존
merged = merged.rename(columns={
    'channel_follower_count': 'channel_follower_count_original',
    'follower_count_new': 'channel_follower_count'
})

# 결과 확인
print('구독자 수 결측:', merged['channel_follower_count'].isna().sum())

display(
    merged[
        ['channel', 'channel_follower_count',
         'follower_collected_at']
    ].drop_duplicates()
)

구독자 수 결측: 0


,channel,channel_follower_count,follower_collected_at
0,1분 샤샤샥,20600,2026-11-09T06:46:18.470732+00:00
90,K드릭스,1620,2026-11-09T06:46:18.470732+00:00
124,드라마저장,28300,2026-11-09T06:46:18.470732+00:00
399,1분드라마모음zip,989000,2026-11-09T06:46:18.470732+00:00
549,드라마 한입,372,2026-11-09T06:46:18.470732+00:00
698,한드TV,94400,2026-11-09T06:46:18.470732+00:00
848,숏까네,37600,2026-11-09T06:46:18.470732+00:00
1403,아워무비,27700,2026-11-09T06:46:18.470732+00:00


In [72]:
display(
    merged.loc[
        merged['drama_guess'].isna(),
        ['video_id', 'title', 'drama_name', 'url', 'source']
    ]
)

,video_id,title,drama_name,url,source
84,0tkNsqYf0cY,이른바 군대병ㅋㅋ,신병3,https://www.youtube.com/shorts/0tkNsqYf0cY,형신


## 컬럼 삭제

In [73]:
# 불필요한 컬럼 삭제
drop_cols = [
    'drama_guess',
    'view_count_original',
    'like_count_original',
    'comment_count_original',
    'channel_follower_count_original',
    'collected_at_original',
    'uploaded_at_new',
    'duration_api',
    'follower_collected_at'
]

merged = merged.drop(columns=drop_cols, errors='ignore')

print('최종 데이터 크기:', merged.shape)
print('남은 컬럼:', merged.columns.tolist())

최종 데이터 크기: (1483, 25)
남은 컬럼: ['video_id', 'url', 'channel', 'channel_id', 'channel_url', 'title', 'description', 'tags', 'title_hashtags', 'desc_hashtags', 'drama_name', 'uploaded_at', 'days_since_upload', 'duration_sec', 'width', 'height', 'vcodec', 'video_file', 'source', 'view_count', 'like_count', 'comment_count', 'collected_at', 'api_found', 'channel_follower_count']


In [74]:
# 1. API 조회 실패 영상 제거
merged = merged[merged['api_found'] == True].copy()

# 2. 불필요한 컬럼 제거
drop_cols = ['api_found']

merged = merged.drop(columns=drop_cols, errors='ignore')
merged = merged.reset_index(drop=True)

## 최종

In [75]:
# 4. 최종 데이터 확인
print('최종 데이터 크기:', merged.shape)
print('조회수 결측:', merged['view_count'].isna().sum())
print('좋아요 결측:', merged['like_count'].isna().sum())
print('댓글 수 결측:', merged['comment_count'].isna().sum())

# 5. 최종 CSV 저장
merged.to_csv(
    save_path / 'shorts_final.csv',
    index=False,
    encoding='utf-8-sig'
)

print('최종 데이터셋 저장 완료!')

최종 데이터 크기: (1468, 24)
조회수 결측: 0
좋아요 결측: 0
댓글 수 결측: 1
최종 데이터셋 저장 완료!


- 댓글 수 결측은 영상 직접 확인해본 결과 댓글 사용이 중지된 상태였음
- 이에 0으로 말고 NaN으로 유지함

In [76]:
missing = pd.DataFrame({
    '결측치 수': merged.isna().sum(),
    '결측 비율(%)': (merged.isna().mean() * 100).round(2)
})

print('\n===== 전체 컬럼 결측치 =====')
display(missing.sort_values('결측치 수', ascending=False))


===== 전체 컬럼 결측치 =====


,결측치 수,결측 비율(%)
tags,1355,92.30
desc_hashtags,301,20.50
title_hashtags,206,14.03
description,110,7.49
height,1,0.07
width,1,0.07
comment_count,1,0.07
video_file,1,0.07
vcodec,1,0.07
title,0,0.00


- 영상 `OtkNsqYfOcY`는 YouTube API를 통해 조회수, 좋아요 수, 댓글 수 등의 메타데이터를 정상적으로 수집했음
- 하지만 `video_file`, `width`, `height`, `vcodec`은 API로 수집할 수 없어 결측치로 남아 있음
- 따라서 해당 영상은 삭제하지 않고 유지했고, 추후 실제 영상 파일 확인 후 결측치를 보완할 예정

> 드라이브 각각의 zip 파일 - data 파일 - video 파일 속 영상에 JSON 원본 파일 & MP4 파일 존재